# Fine-tune LLaMA 3.2 3B for Ido↔Esperanto (QLoRA / Unsloth)

One bidirectional model on the dataset built by `llm/data/make_dataset.py`,
pulled from GitHub. Runs on **Kaggle** (disconnect-proof) or Colab.

## Kaggle — recommended (survives closing the browser)
1. kaggle.com → **Create → New Notebook → File → Import Notebook** → paste this
   notebook's URL (`https://github.com/komapc/ido-epo-llm/blob/main/finetune_colab.ipynb`)
   or upload the `.ipynb`.
2. Right sidebar **Settings**: Accelerator = **GPU T4 x2** (or P100);
   **Internet = On** (needs a phone-verified account — required for pip + git clone).
3. **Save Version → Save & Run All (Commit)**. It runs to completion on Kaggle's
   servers — you can close the tab. When it finishes, open that version →
   **Output** tab → download `preds.jsonl` and `ido-epo-lora.zip` (written to
   `/kaggle/working/`).

Data repo is **public**, so no token/secret is needed.

## Colab — alternative
Runtime → Change runtime type → T4 GPU → Run all. (Free Colab may disconnect on
long unattended runs; prefer the Kaggle commit.)

Baseline to beat (Apertium on the real test split): **chrF 57.0 / BLEU 18.1**,
especially the 60% of inputs where Apertium emits a `*`/`#`/`@` failure marker.

In [ ]:
%%capture
# Unsloth = fast 4-bit QLoRA, fits 3B on a free T4.
# IMPORTANT: upgrade BOTH unsloth and unsloth_zoo together. Kaggle's image can
# ship a stale unsloth_zoo, which breaks the patched training step with
# "'int' object has no attribute 'mean'". Upgrading the pair (and letting it
# pull a compatible transformers/trl) fixes it.
# Do NOT reinstall `trl --no-deps` separately — that causes a different
# SFTConfig PicklingError at checkpoint-save time.
!pip install -q --upgrade --no-cache-dir unsloth unsloth_zoo

In [ ]:
# --- Pull dataset from the repo ---------------------------------------------
import os, subprocess

REPO_URL    = "https://github.com/komapc/ido-epo-llm.git"  # change to your repo
DATA_SUBDIR = "data"          # path inside the repo holding train/val/test.jsonl
BRANCH      = "main"
# For a PRIVATE repo, set a fine-grained PAT (Colab: 🔑 Secrets -> GH_TOKEN):
GH_TOKEN    = os.environ.get("GH_TOKEN", "")
try:
    from google.colab import userdata
    GH_TOKEN = GH_TOKEN or userdata.get("GH_TOKEN")
except Exception:
    pass

clone_url = REPO_URL
if GH_TOKEN:
    clone_url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@")

REPO_DIR = REPO_URL.rstrip('/').split('/')[-1][:-4]  # 'ido-epo-llm'
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, clone_url], check=True)

DATA_DIR = os.path.join(REPO_DIR, DATA_SUBDIR)
for split in ("train", "val", "test"):
    p = os.path.join(DATA_DIR, f"{split}.jsonl")
    assert os.path.exists(p), f"missing {p} — check REPO_URL/DATA_SUBDIR"
    print(split, sum(1 for _ in open(p)), "rows")

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 512
BASE = "unsloth/Llama-3.2-3B-Instruct"  # path to 8B: swap for unsloth/Meta-Llama-3.1-8B-Instruct

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
import json
from datasets import Dataset

def load_jsonl(name):
    path = os.path.join(DATA_DIR, name)
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

train_rows = load_jsonl("train.jsonl")

# Oversample by integer weight so real pairs (w=3) are seen 3x vs synthetic (w=1).
expanded = []
for r in train_rows:
    expanded.extend([r] * int(r.get("weight", 1)))
print(f"{len(train_rows)} unique -> {len(expanded)} after weighting")

def to_text(r):
    msgs = [
        {"role": "system", "content": "You are a precise Ido↔Esperanto translator. Output only the translation."},
        {"role": "user", "content": f"{r['instruction']}\n{r['input']}"},
        {"role": "assistant", "content": r["output"]},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False)

ds = Dataset.from_list([{"text": to_text(r)} for r in expanded]).shuffle(seed=42)
print(ds[0]["text"][:400])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# 1 epoch: loss already plateaus (~0.23) by end of epoch 1, and the real pairs
# are oversampled 3x, so a 2nd epoch mostly memorizes. Bump to 2 only if eval
# says it's underfitting. ~2.5h on a T4.
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    dataset_text_field="text", max_seq_length=MAX_SEQ, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=8, gradient_accumulation_steps=2,
        warmup_ratio=0.03, num_train_epochs=1, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="cosine", seed=42, output_dir="outputs",
        save_strategy="epoch", save_only_model=True, report_to="none",
    ),
)
trainer.train()

In [ ]:
# Generate predictions on the held-out real test split -> preds.jsonl
FastLanguageModel.for_inference(model)
test_rows = load_jsonl("test.jsonl")

def translate(instruction, text):
    msgs = [
        {"role": "system", "content": "You are a precise Ido↔Esperanto translator. Output only the translation."},
        {"role": "user", "content": f"{instruction}\n{text}"},
    ]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=128, do_sample=False, temperature=0.0)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

with open("preds.jsonl", "w", encoding="utf-8") as f:
    for i, r in enumerate(test_rows):
        pred = translate(r["instruction"], r["input"])
        f.write(json.dumps({"output": pred}, ensure_ascii=False) + "\n")
        if i < 8:
            print(f"{r['input']}\n  ref : {r['output']}\n  pred: {pred}\n")
print("wrote preds.jsonl — download it and run: python3 eval/eval_chrf.py --apertium --pred preds.jsonl")

In [ ]:
# Save the LoRA adapter (small). Optionally push to HF Hub or save merged GGUF.
model.save_pretrained("ido-epo-lora")
tokenizer.save_pretrained("ido-epo-lora")
!zip -r ido-epo-lora.zip ido-epo-lora >/dev/null && echo 'adapter saved to ido-epo-lora.zip'

# To serve later: merge to 16-bit or export GGUF/vLLM, then wire the worker engine flag.
# model.save_pretrained_merged("ido-epo-merged", tokenizer, save_method="merged_16bit")
# model.push_to_hub_gguf("komapc/ido-epo-llama-3.2-3b", tokenizer, quantization_method="q4_k_m", token=...)